# Data Science Fundamentals Assessment
**Internship Track:** Data Science Core  
**Topic:** Comprehensive Practical Implementation Covering Python, Statistics, Data Types, and Exploratory Data Analysis  
**Author:** Data Science Intern  
---
### Project Overview & Objectives
1. **Understand Data Science Fundamentals:** Lifecycle, analysis types, industry use cases.
2. **Apply Python Programming Basics:** Variables, loops, functions, lists, dicts, file handling.
3. **Learn Statistical Foundations:** Mean, median, mode, variance, std dev, percentiles, correlation.
4. **Understand Data Types & Structures:** Numerical, categorical, ordinal, datetime, text.
5. **Follow Best Practices:** Clean code, PEP 8, reproducible random seeds, documentation.
6. **Complete Practical Assignments:** Empirical data generation, analysis, and visualization.

## 1. Environment Setup & Dependency Ingestion

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set reproducible random seed
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries loaded successfully.")

## 2. Multi-Modal Data Generation & Python Programming Primitives
We create a realistic synthetic customer banking dataset representing **250 customer profiles** spanning multiple data types:
- **Numerical Continuous:** `AnnualIncome`
- **Numerical Discrete:** `Age`, `CreditScore`
- **Categorical Nominal:** `Department`
- **Categorical Ordinal:** `SatisfactionLevel`
- **Temporal Datetime:** `SignupDate`
- **Unstructured Text:** `CustomerFeedback`
- **Binary Indicator:** `Churned`

In [2]:
n_samples = 250

customer_ids = [f"CUST_{1000 + i}" for i in range(n_samples)]
ages = np.random.randint(18, 70, size=n_samples)
incomes = np.round(np.clip(np.random.normal(loc=55000, scale=15000, size=n_samples), 20000, 120000), 2)
credit_scores = np.clip(np.random.normal(loc=680, scale=60, size=n_samples).astype(int), 300, 850)

satisfaction_levels = ["Low", "Medium", "High", "Very High"]
satisfaction = np.random.choice(satisfaction_levels, size=n_samples, p=[0.15, 0.35, 0.35, 0.15])

departments = ["Retail", "Corporate", "SME", "Private Banking"]
department = np.random.choice(departments, size=n_samples, p=[0.40, 0.25, 0.20, 0.15])

start_date = pd.to_datetime("2023-01-01")
random_days = np.random.randint(0, 365, size=n_samples)
signup_dates = [start_date + pd.Timedelta(days=int(d)) for d in random_days]

feedbacks = [
    "Excellent customer service and transparent charges.",
    "Mobile app response is sluggish, need urgent bug fixes.",
    "Great interest rates on savings accounts and smooth onboarding.",
    "Customer support phone line was busy for over twenty minutes.",
    "Prompt transaction confirmation and easy foreign remittances."
]
customer_feedback = np.random.choice(feedbacks, size=n_samples)

# Churn probability calculation based on features
churn_flags = []
for i in range(n_samples):
    p = 0.2
    if satisfaction[i] == "Low":
        p += 0.4
    elif satisfaction[i] == "Medium":
        p += 0.1
    if credit_scores[i] < 620:
        p += 0.2
    churn_flags.append(int(np.random.binomial(1, min(0.95, p))))

df = pd.DataFrame({
    "CustomerID": customer_ids,
    "Age": ages,
    "AnnualIncome": incomes,
    "Department": department,
    "SatisfactionLevel": satisfaction,
    "CreditScore": credit_scores,
    "SignupDate": signup_dates,
    "CustomerFeedback": customer_feedback,
    "Churned": churn_flags
})

df.head()

## 3. Statistical Foundations: Central Tendency, Dispersion & Correlation
Computing parametric and non-parametric summary statistics:

In [3]:
def compute_statistics(series, label):
    mean_val = np.mean(series)
    median_val = np.median(series)
    vals, cnts = np.unique(series, return_counts=True)
    mode_val = vals[np.argmax(cnts)]
    std_val = np.std(series, ddof=1)
    var_val = np.var(series, ddof=1)
    q25, q75 = np.percentile(series, [25, 75])
    iqr_val = q75 - q25
    skew_val = pd.Series(series).skew()
    
    return {
        "Feature": label,
        "Mean": round(mean_val, 2),
        "Median": round(median_val, 2),
        "Mode": round(mode_val, 2),
        "Variance": round(var_val, 2),
        "Std Dev": round(std_val, 2),
        "Q1 (25%)": round(q25, 2),
        "Q3 (75%)": round(q75, 2),
        "IQR": round(iqr_val, 2),
        "Skewness": round(skew_val, 2)
    }

stat_summary = pd.DataFrame([
    compute_statistics(df["AnnualIncome"], "Annual Income ($)"),
    compute_statistics(df["CreditScore"], "Credit Score"),
    compute_statistics(df["Age"], "Age (Years)")
])

stat_summary

### Correlation Analysis (Pearson vs Spearman)

In [4]:
p_corr = df["AnnualIncome"].corr(df["CreditScore"], method="pearson")
s_corr = df["AnnualIncome"].corr(df["CreditScore"], method="spearman")

print(f"Pearson Correlation (Linear): {p_corr:.4f}")
print(f"Spearman Correlation (Monotonic Rank): {s_corr:.4f}")

## 4. Visualizations: Distributions, Boxplots & Heatmaps

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Income Distribution
sns.histplot(df["AnnualIncome"], kde=True, ax=axes[0], color="#2563EB", bins=20)
axes[0].set_title("Annual Income Distribution", fontsize=12, fontweight="bold")
axes[0].axvline(df["AnnualIncome"].mean(), color="#DC2626", linestyle="--", label=f'Mean: ${df["AnnualIncome"].mean():,.0f}')
axes[0].axvline(df["AnnualIncome"].median(), color="#16A34A", linestyle="-", label=f'Median: ${df["AnnualIncome"].median():,.0f}')
axes[0].legend()

# Credit Score by Satisfaction Level
sns.boxplot(x="SatisfactionLevel", y="CreditScore", data=df, order=["Low", "Medium", "High", "Very High"], ax=axes[1], palette="Blues_r", hue="SatisfactionLevel", legend=False)
axes[1].set_title("Credit Score by Satisfaction Level (Ordinal Structure)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of Numerical Features
num_cols = ["Age", "AnnualIncome", "CreditScore", "Churned"]
sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f", ax=axes[0], linewidths=1)
axes[0].set_title("Pearson Correlation Heatmap", fontsize=12, fontweight="bold")

# Churn Rate by Department
dept_churn = df.groupby("Department")["Churned"].mean().reset_index()
sns.barplot(x="Department", y="Churned", data=dept_churn, ax=axes[1], palette="crest", hue="Department", legend=False)
axes[1].set_title("Customer Churn Rate by Department", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Churn Proportion")

plt.tight_layout()
plt.show()

## 5. Conclusions & Key Findings
- **Data Science Lifecycle & Typing:** Managing numerical, ordinal, nominal, temporal, and unstructured text types requires tailored scaling and encoding treatments.
- **Statistical Properties:** The median provides robust resistance to outliers in income distributions, while credit scores exhibit normal bell-shaped traits.
- **Engineering Best Practices:** Deterministic seed setting, PEP 8 compliance, modular functional decomposition, and comprehensive documentation ensure reproducible analysis.